# Test the Odia normalizer in Google Colab
**Author: Sambit Sekhar**

Use a fresh **CPU runtime** and choose **Runtime → Run all**. No GPU or API key
is needed. Setup downloads the source, installs Rust, and compiles the Python
extension; the first run takes several minutes and requires internet access.
The examples below have assertions and stop if an expected result fails.

This is an alpha text normalizer. Vocabulary still needs native-speaker/audio
review. The package is installed from a pinned GitHub revision because it is
not yet published to PyPI. Supported Python versions: CPython 3.11–3.14.

## 1. Install from GitHub
Setup uses the same Python interpreter as the notebook kernel. It keeps the
source checkout under `/content/normalizer-or` and does not mount Google Drive.
Run setup before importing `normalizer_or`; after changing the source revision,
start a fresh runtime to avoid retaining a loaded native extension.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.request

assert (3, 11) <= sys.version_info[:2] < (3, 15), sys.version
repository = "https://github.com/OliveSuite-AI/normalizer-or.git"
revision = "de0a07d"  # Tested source revision with the full Python example suite.
repo_dir = Path("/content/normalizer-or")

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--quiet", repository, str(repo_dir)], check=True)
    subprocess.run(["git", "checkout", "--detach", revision], cwd=repo_dir, check=True)
else:
    # Do not overwrite files from a previous or unrelated checkout.
    actual = subprocess.check_output(
        ["git", "rev-parse", "--short=7", "HEAD"], cwd=repo_dir, text=True
    ).strip()
    if actual != revision:
        raise RuntimeError(
            "Checkout differs from the requested revision. Start a fresh Colab runtime."
        )

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ["PATH"]
if not shutil.which("rustup"):
    with tempfile.TemporaryDirectory() as directory:
        installer = Path(directory) / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", installer)
        subprocess.run(
            [
                "sh",
                str(installer),
                "-y",
                "--profile",
                "minimal",
                "--default-toolchain",
                "none",
            ],
            check=True,
        )
subprocess.run(
    ["rustup", "toolchain", "install", "1.99.0", "--profile", "minimal"], check=True
)
os.environ["RUSTUP_TOOLCHAIN"] = "1.99.0"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(repo_dir / "bindings/python"),
    ],
    check=True,
)
print("Installed from", revision, "using", sys.executable)

## 2. Quick start
`normalize` expands supported expressions into speech text. `clean` performs
Unicode/digit/whitespace cleanup. The result reports completeness and source
spans; it is not just a string.

In [ ]:
from normalizer_or import BUILD_VERSION, NORMALIZER_ID, Normalizer

n = Normalizer()
result = n.normalize("ଆଜି ୨୫ kg ଚାଉଳ।")
assert result.normalized_text == "ଆଜି ପଚିଶ କିଲୋଗ୍ରାମ ଚାଉଳ।"
assert result.complete
print("Build:", BUILD_VERSION, NORMALIZER_ID)
print(result.normalized_text)

## 3. Numbers, money, percentages, units, dates and time
Change a source value to experiment. Update an expected value only after
independently reviewing its Odia reading.

In [ ]:
cases = [
    ("1୨3", "ଏକ ଶହ ତେଇଶ"),
    ("1,00,000", "ଏକ ଲକ୍ଷ"),
    ("1,00,00,000", "ଏକ କୋଟି"),
    ("12.050", "ବାର ଦଶମିକ ଶୂନ୍ୟ ପାଞ୍ଚ ଶୂନ୍ୟ"),
    ("₹25.50", "ପଚିଶ ଟଙ୍କା ପଚାଶ ପଇସା"),
    ("12.5%", "ବାର ଦଶମିକ ପାଞ୍ଚ ପ୍ରତିଶତ"),
    ("୫ kg", "ପାଞ୍ଚ କିଲୋଗ୍ରାମ"),
    ("25°C", "ପଚିଶ ଡିଗ୍ରୀ ସେଲସିୟସ"),
    ("23/10/2026", "ତେଇଶ ଅକ୍ଟୋବର ଦୁଇ ହଜାର ଛବିଶ"),
    ("09:30", "ନଅ ଘଣ୍ଟା ତିରିଶ ମିନିଟ"),
]
for source, expected in cases:
    result = n.normalize(source)
    assert result.normalized_text == expected and result.complete
    print(source, "→", result.normalized_text)
print("PASS:", len(cases), "speech cases")

## 4. Written-text cleanup
All digit and whitespace options are demonstrated here. Collapsing whitespace
retains one space at each input edge. Indic joiners are preserved.

In [ ]:
source = "  କୋ କ୍\u200dଷ କ୍\u200cଷ\n୧2  "
for digits, tail in [("preserve", "୧2"), ("ascii", "12"), ("odia", "୧୨")]:
    for whitespace in ["preserve", "collapse"]:
        result = n.clean(source, digits=digits, whitespace=whitespace)
        expected = f"  କୋ କ୍\u200dଷ କ୍\u200cଷ\n{tail}  "
        if whitespace == "collapse":
            expected = f" କୋ କ୍\u200dଷ କ୍\u200cଷ {tail} "
        assert result.normalized_text == expected
        assert (
            n.clean(expected, digits=digits, whitespace=whitespace).normalized_text
            == expected
        )
        print(digits, whitespace, repr(result.normalized_text))

## 5. Preserve, reject or use fallback
Malformed expressions remain whole. Fallback reads the literal source and
reports an assumption; it does not repair invalid dates or validate identifiers.

In [ ]:
from normalizer_or import NormalizationError

for source in ["1,,234", "₹25.005", "29/02/2025", "AB12"]:
    kept = n.normalize(source, ambiguity_policy="preserve")
    assert kept.normalized_text == source and not kept.complete
    try:
        n.normalize(source, ambiguity_policy="reject")
    except NormalizationError as exc:
        assert exc.code == "unresolved" and exc.issues == kept.issues
    else:
        raise AssertionError("Expected strict rejection")
    fallback = n.normalize(source, ambiguity_policy="fallback")
    assert fallback.complete and fallback.fallback_used
    print(source, "→", fallback.normalized_text, fallback.fallbacks)

## 6. Hints and UTF-8 source offsets
Offsets count UTF-8 bytes. `len(text)` counts Python characters and cannot be
used as an Odia byte offset. A cardinal hint overrides a leading-zero reading.

In [ ]:
from normalizer_or import Hint

prefix = "ଓଡ଼ିଆ "
source = prefix + "୦୧୨"
start = len(prefix.encode("utf-8"))
end = len(source.encode("utf-8"))
assert n.normalize(source).normalized_text == prefix + "ଶୂନ୍ୟ ଏକ ଦୁଇ"
result = n.normalize(source, hints=[Hint(start, end, "cardinal")])
assert result.normalized_text == prefix + "ବାର"
print(result.normalized_text)

source = "ଓଡ଼ିଆ ୨୫ kg; AB12; 24:00"
result = n.normalize(source)
encoded = source.encode("utf-8")
assert len(result.issues) == 2
for issue in result.issues:
    original = encoded[issue.start_byte : issue.end_byte].decode("utf-8")
    print(original, issue.category, (issue.start_byte, issue.end_byte))
assert "".join(segment.text for segment in result.segments) == result.normalized_text

## 7. JSON results and errors
The immutable result is a dataclass. Use `ensure_ascii=False` to retain readable
Odia in JSON. Empty input raises `invalid_input`. Cancellation remains an error
under every ambiguity policy. Deadlines are cooperative, so short examples do
not assume that a timeout will occur.

In [ ]:
import json
from dataclasses import asdict
from normalizer_or import CancellationToken

record = asdict(n.normalize("୨୫ kg"))
print(json.dumps(record, ensure_ascii=False, indent=2))
assert json.loads(json.dumps(record))["normalized_text"] == "ପଚିଶ କିଲୋଗ୍ରାମ"

for source in ["", "a" * 32769]:
    try:
        n.normalize(source)
    except NormalizationError as exc:
        assert exc.code == ("invalid_input" if not source else "limit_exceeded")
        print("Expected error:", exc.code, exc.limit_kind)
    else:
        raise AssertionError("Expected input error")

token = CancellationToken()
token.cancel()
try:
    n.normalize("୫", cancellation=token)
except NormalizationError as exc:
    assert exc.code == "cancelled"
    print("Expected error:", exc.code)
else:
    raise AssertionError("Expected cancellation")
assert n.normalize("୫", deadline_ms=60000).complete

## 8. Run all seven example scripts
This also covers all four hint kinds, explicit phone digit hints, every supported
unit, threaded batch processing, UTF-8 JSONL output, and invalid option handling.
A failed script stops execution with a nonzero exit status.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, str(repo_dir / "bindings/python/examples/run_all.py")],
    check=True,
    timeout=120,
)
print("PASS: notebook examples and standalone suite completed")

## 9. Try your own input
Edit `text` and `policy`, then rerun this cell. Preserve is a useful default when
you want to inspect unsupported expressions before choosing a fallback.

In [ ]:
text = "ଆଜି ୨୫ kg ଚାଉଳ। ₹25.50"
policy = "preserve"  # preserve, reject, or fallback
try:
    result = n.normalize(text, ambiguity_policy=policy)
    print(result.normalized_text)
    print("Complete:", result.complete, "Fallback:", result.fallback_used)
    print("Issues:", result.issues)
except NormalizationError as exc:
    print("Error:", exc.code, exc.issues)

References: [repository](https://github.com/OliveSuite-AI/normalizer-or),
[example guide](https://github.com/OliveSuite-AI/normalizer-or/blob/main/bindings/python/examples/README.md),
[Colab GitHub support](https://github.com/googlecolab/open_in_colab),
[Rust installation](https://rust-lang.org/tools/install/).